<div class="blog-language-switch" role="group" aria-label="Article language">
<span aria-current="page">English</span>
<a href="../zh-CN/Deep-Learning/09-attention-transformers.html" lang="zh-CN" hreflang="zh-CN">中文</a>
</div>

[Back to Deep Learning guideline](Deep-Learning.html)

## **Attention and Transformers** {#attention-transformers}

Chapter 08 compared recurrence, temporal convolution, and state-space compression. Each method moves information through a predetermined state transition or local kernel. Attention introduces a different operation: a position constructs its representation by **content-addressing** a collection of other positions. The Transformer organizes this operation with projections, residual paths, normalization, feed-forward networks, and positional information so that an entire sequence can be trained in parallel.

The chapter continues the [OpenSLR SLR1 YESNO corpus](https://openslr.org/1/) used in Chapter 08. All examples operate on the same locally stored 8 kHz recordings and their eight yes/no labels. Waveforms become normalized log-STFT frames, then every fourth frame is retained to keep the quadratic teaching examples inexpensive. This temporal subsampling is part of the declared preprocessing, not an invisible optimization. The dataset contains only 60 utterances from one male speaker, so predictive scores are mechanism checks rather than broad speech benchmarks. The official [torchaudio `YESNO` documentation](https://docs.pytorch.org/audio/main/generated/torchaudio.datasets.YESNO.html) points to the same OpenSLR archive.

### **Why Attention Was Introduced** {#why-attention-was-introduced}

Early encoder-decoder RNNs compressed an entire source sequence into one final hidden vector. When the source became long, that vector had to preserve every detail needed by every decoder step. Attention replaced the single bottleneck with a set of encoder states and let each output step retrieve a different weighted combination. The retrieval path from any source position to the current output became short, and the model could expose a soft alignment.

Self-attention applies the same principle within one sequence. It replaces the fixed neighborhood of convolution and the step-by-step path of recurrence with data-dependent interactions. For length $T$ and hidden dimension $D$, full self-attention computes a $T\times T$ score matrix. The short dependency path and parallel training are powerful, but time and memory scale quadratically with $T$ unless structure or an optimized memory schedule is used.

Attention is not a memory system by itself. It defines how one set of representations reads another. A Transformer still needs input embeddings, position or geometry, nonlinear channel mixing, residual optimization paths, and a task head. The quality of an attention map also depends on learned projections; a heatmap is not automatically a faithful explanation of the final prediction.

<details>
<summary><strong>PyTorch: establish the shared YESNO attention experiment</strong></summary>

```python
import io
import math
import random
import tarfile
import wave
from pathlib import Path

import numpy as np
import torch
from sklearn.model_selection import train_test_split
from torch import nn
from torch.nn import functional as F
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import DataLoader, Dataset

torch.set_num_threads(1)


def seed_everything(seed=909):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


def read_pcm_wave(payload):
    with wave.open(io.BytesIO(payload), "rb") as stream:
        assert stream.getnchannels() == 1 and stream.getsampwidth() == 2
        sample_rate = stream.getframerate()
        samples = np.frombuffer(stream.readframes(stream.getnframes()), dtype="<i2").copy()
    return torch.tensor(samples, dtype=torch.float32) / 32768.0, sample_rate


archive_candidates = [
    Path("assets/data/waves_yesno.tar.gz"),
    Path("ipynb/Deep-Learning/assets/data/waves_yesno.tar.gz"),
]
archive = next(path for path in archive_candidates if path.exists())
window = torch.hann_window(256)
records = []
with tarfile.open(archive, "r:gz") as bundle:
    members = sorted(
        (member for member in bundle.getmembers() if member.isfile() and member.name.endswith(".wav")),
        key=lambda member: member.name,
    )
    for member in members:
        waveform, sample_rate = read_pcm_wave(bundle.extractfile(member).read())
        spectrum = torch.stft(
            waveform, n_fft=256, hop_length=160, win_length=256,
            window=window, return_complex=True,
        ).abs().transpose(0, 1)
        features = torch.log1p(spectrum)[::4]  # 20 ms hop becomes an 80 ms teaching sequence.
        labels = torch.tensor([int(value) for value in Path(member.name).stem.split("_")])
        records.append({"name": member.name, "features": features, "labels": labels})

all_idx = np.arange(len(records))
first_labels = np.array([int(record["labels"][0]) for record in records])
train_idx, holdout_idx = train_test_split(
    all_idx, test_size=0.30, random_state=808, stratify=first_labels
)
val_idx, test_idx = train_test_split(
    holdout_idx, test_size=0.50, random_state=808, stratify=first_labels[holdout_idx]
)

training_frames = torch.cat([records[int(i)]["features"] for i in train_idx])
feature_mean = training_frames.mean(0)
feature_std = training_frames.std(0).clamp_min(1e-5)
for record in records:
    record["features"] = (record["features"] - feature_mean) / feature_std


class YesNoDataset(Dataset):
    def __init__(self, indices):
        self.indices = [int(i) for i in indices]

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, index):
        return records[self.indices[index]]


def collate_yesno(batch):
    sequences = [item["features"] for item in batch]
    lengths = torch.tensor([len(sequence) for sequence in sequences])
    padded = pad_sequence(sequences, batch_first=True)
    valid = torch.arange(padded.shape[1]).unsqueeze(0) < lengths.unsqueeze(1)
    labels = torch.stack([item["labels"] for item in batch])
    return padded, lengths, valid, labels


def yesno_loader(indices, shuffle=False, seed=909, batch_size=6):
    generator = torch.Generator().manual_seed(seed)
    return DataLoader(
        YesNoDataset(indices), batch_size=batch_size, shuffle=shuffle,
        generator=generator, collate_fn=collate_yesno,
    )


acoustic_frames, frame_lengths, frame_mask, word_labels = next(iter(yesno_loader(train_idx)))
assert len(records) == 60 and sample_rate == 8000
assert acoustic_frames.shape[0] == 6 and acoustic_frames.shape[-1] == 129
assert frame_mask.sum(1).equal(frame_lengths)
assert set(train_idx).isdisjoint(test_idx)
print({"split": (len(train_idx), len(val_idx), len(test_idx)),
       "frames": tuple(acoustic_frames.shape),
       "valid length range": (int(frame_lengths.min()), int(frame_lengths.max()))})
```

</details>

Every later example reuses this split, feature normalization, and padding mask. A new attention mechanism is therefore tested against the same sequence geometry rather than a newly generated tensor that happens to have convenient dimensions.

### **Queries, Keys, and Values** {#queries-keys-values}

Attention separates three roles. A **query** describes what the current position wants to retrieve. A **key** describes what each candidate position offers for matching. A **value** contains the information returned when that candidate receives weight. Given query $q$, key-value pairs $(k_i,v_i)$, and compatibility function $a$, attention pooling is

$$
\alpha_i=\frac{\exp(a(q,k_i))}{\sum_j\exp(a(q,k_j))},
\qquad
\operatorname{Attention}(q,K,V)=\sum_i\alpha_i v_i.
$$

Keys and values belong to the same candidate positions but need not have the same representation. A frame may be useful to locate because of one acoustic pattern while returning a different transformed feature. Learned projections make these roles task dependent:

$$
Q=XW_Q,\qquad K=XW_K,\qquad V=XW_V.
$$

The row-wise softmax makes $\alpha_i\ge0$ and $\sum_i\alpha_i=1$, so each output is a convex combination of value vectors. The projections and later residual/FFN layers can still create signed, nonlinear transformations; the convexity applies only to one attention aggregation.

![A query compares with keys to produce weights that pool the associated values.](assets/dl09-qkv.svg){fig-align="center" width="72%" fig-alt="Query, key, and value diagram in which compatibility scores become attention weights over values."}

*Image source: [Dive into Deep Learning, Queries, Keys, and Values](https://d2l.ai/chapter_attention-mechanisms-and-transformers/queries-keys-values.html), CC BY-SA 4.0.*

<details>
<summary><strong>PyTorch: retrieve context from one real acoustic sequence</strong></summary>

```python
seed_everything(910)
model_dimension = 32
query_projection = nn.Linear(129, model_dimension, bias=False)
key_projection = nn.Linear(129, model_dimension, bias=False)
value_projection = nn.Linear(129, model_dimension, bias=False)

sequence = acoustic_frames[0, :frame_lengths[0]]
query_frame = sequence[len(sequence) // 2:len(sequence) // 2 + 1]
query = query_projection(query_frame)                    # [1, D]
keys = key_projection(sequence)                          # [T, D]
values = value_projection(sequence)                      # [T, D]
scores = query @ keys.transpose(0, 1) / math.sqrt(model_dimension)
weights = scores.softmax(-1)
context = weights @ values

top_positions = weights[0].topk(3).indices.tolist()
assert context.shape == (1, model_dimension)
assert torch.allclose(weights.sum(-1), torch.ones(1))
assert max(top_positions) < len(sequence)
print({"query position": len(sequence) // 2,
       "top key positions": top_positions,
       "context shape": tuple(context.shape)})
```

</details>

The untrained projections make the selected positions arbitrary; the code verifies retrieval semantics, not interpretability. After task training, weights become part of a learned computation, but causal attribution still requires intervention or gradient analysis rather than reading one heatmap literally.

### **Scaled Dot-Product Attention** {#scaled-dot-product-attention}

For batched matrices $Q\in\mathbb{R}^{T_q\times d_k}$, $K\in\mathbb{R}^{T_k\times d_k}$, and $V\in\mathbb{R}^{T_k\times d_v}$, scaled dot-product attention is

$$
\operatorname{Attention}(Q,K,V)
=\operatorname{softmax}\!\left(\frac{QK^\top}{\sqrt{d_k}}+M\right)V.
$$

$M$ is an additive mask: permitted entries receive zero and forbidden entries receive $-\infty$ before softmax. If independent query and key components have variance one, their dot product has variance approximately $d_k$. Dividing by $\sqrt{d_k}$ keeps score variance near one, preventing the softmax from becoming increasingly saturated as head dimension grows.

Numerical stability requires subtracting each row maximum before exponentiation. Framework functions combine this log-sum-exp transformation internally. In reduced precision, directly constructing a very negative finite constant can overflow or behave differently across dtypes; boolean masks passed to a maintained scaled-dot-product attention primitive are preferable.

The $T_qT_k$ scores dominate full attention memory. Output storage is only $T_qd_v$, but a naive implementation materializes both scores and probabilities. FlashAttention later changes that memory schedule while preserving this exact equation.

<details>
<summary><strong>PyTorch: verify the scale and stable softmax on YESNO frames</strong></summary>

```python
tokens = query_projection(acoustic_frames[:2])
queries = tokens
keys_for_scale = key_projection(acoustic_frames[:2])
raw_scores = queries @ keys_for_scale.transpose(-2, -1)
scaled_scores = raw_scores / math.sqrt(model_dimension)

stable_probabilities = torch.exp(scaled_scores - scaled_scores.amax(-1, keepdim=True))
stable_probabilities = stable_probabilities / stable_probabilities.sum(-1, keepdim=True)
library_probabilities = scaled_scores.softmax(-1)

raw_variance = raw_scores.var().item()
scaled_variance = scaled_scores.var().item()
assert torch.allclose(stable_probabilities, library_probabilities, atol=1e-6)
assert scaled_variance < raw_variance
assert torch.allclose(library_probabilities.sum(-1), torch.ones_like(library_probabilities[..., 0]))
print({"raw score variance": raw_variance, "scaled score variance": scaled_variance,
       "score elements": raw_scores.numel()})
```

</details>

Scaling controls the initial score distribution; it does not guarantee well-behaved attention after training. Query/key norms can still grow, so normalization, initialization, regularization, and monitoring attention entropy remain relevant.

### **Multi-Head Attention and Tensor Shapes** {#multi-head-attention-tensor-shapes}

One attention map uses one learned similarity space. Multi-head attention creates $H$ subspaces, applies attention independently, concatenates the results, and projects back to model dimension:

$$
\operatorname{head}_h
=\operatorname{Attention}(QW_Q^{(h)},KW_K^{(h)},VW_V^{(h)}),
$$

$$
\operatorname{MHA}(Q,K,V)
=\operatorname{Concat}(\operatorname{head}_1,\ldots,\operatorname{head}_H)W_O.
$$

For $D=Hd_h$, the central shape transformation is

$$
[B,T,D]
\rightarrow[B,T,H,d_h]
\rightarrow[B,H,T,d_h].
$$

The score tensor is `[B,H,T_q,T_k]`. Heads do not automatically learn distinct linguistic or acoustic roles; symmetry can produce redundant heads. Diversity emerges only when the task and optimization make different subspaces useful. Head count also affects hardware efficiency because a very small $d_h$ can create poorly utilized matrix operations.

Parameter count remains roughly $4D^2$ for equal query/key/value dimensions: three projections plus the output projection. Increasing head count at fixed $D$ changes factorization, not this leading parameter count.

<details>
<summary><strong>PyTorch: implement multi-head attention with explicit shape checks</strong></summary>

```python
class ManualMultiHeadAttention(nn.Module):
    def __init__(self, dimension=32, heads=4):
        super().__init__()
        assert dimension % heads == 0
        self.heads = heads
        self.head_dimension = dimension // heads
        self.qkv = nn.Linear(dimension, 3 * dimension, bias=False)
        self.output = nn.Linear(dimension, dimension, bias=False)
        self.last_weights = None

    def split_heads(self, x):
        batch, time, dimension = x.shape
        return x.view(batch, time, self.heads, self.head_dimension).transpose(1, 2)

    def forward(self, x, valid_mask):
        q, k, v = self.qkv(x).chunk(3, dim=-1)
        q, k, v = map(self.split_heads, (q, k, v))
        scores = q @ k.transpose(-2, -1) / math.sqrt(self.head_dimension)
        scores = scores.masked_fill(~valid_mask[:, None, None, :], float("-inf"))
        weights = scores.softmax(-1)
        context = weights @ v
        context = context.transpose(1, 2).contiguous().view(x.shape)
        self.last_weights = weights
        return self.output(context).masked_fill(~valid_mask.unsqueeze(-1), 0.0)


input_projection = nn.Linear(129, model_dimension)
acoustic_tokens = input_projection(acoustic_frames)
manual_mha = ManualMultiHeadAttention(model_dimension, heads=4)
multihead_output = manual_mha(acoustic_tokens, frame_mask)

assert multihead_output.shape == acoustic_tokens.shape
assert manual_mha.last_weights.shape == (
    acoustic_frames.shape[0], 4, acoustic_frames.shape[1], acoustic_frames.shape[1]
)
assert torch.all(multihead_output[~frame_mask] == 0)
print({"tokens": tuple(acoustic_tokens.shape),
       "heads": tuple(manual_mha.last_weights.shape),
       "head dimension": manual_mha.head_dimension})
```

</details>

Calling `.contiguous()` before `.view()` is not cosmetic: transposition changes strides, so the desired logical concatenation may not occupy contiguous memory. Shape and stride reasoning are part of correct attention implementation.

### **Self-Attention and Cross-Attention** {#self-attention-cross-attention}

In **self-attention**, queries, keys, and values originate from the same sequence. Each acoustic frame can therefore gather evidence from other frames. The operation is permutation equivariant if no positional information or mask is added: permuting input tokens permutes outputs in the same way. Sequence order must enter separately.

In **cross-attention**, queries come from one sequence and keys/values from another. A decoder token can query encoder memory, a text token can query image patches, or a small latent array can query a large sensor stream. If query length is $T_q$ and memory length is $T_k$, interaction cost is $O(T_qT_kD)$ rather than necessarily $O(T_k^2D)$.

The roles are asymmetric. Cross-attention output has the query sequence length, because there is one result per query. Padding masks usually describe invalid key/value memory positions; invalid query positions must be handled separately in the output or loss.

<details>
<summary><strong>PyTorch: let eight label queries read acoustic memory</strong></summary>

```python
self_attention = nn.MultiheadAttention(model_dimension, num_heads=4, batch_first=True)
self_output, self_weights = self_attention(
    acoustic_tokens, acoustic_tokens, acoustic_tokens,
    key_padding_mask=~frame_mask,
)

label_embedding = nn.Embedding(3, model_dimension)  # labels 0/1 plus start token 2
label_queries = label_embedding(word_labels)
cross_attention = nn.MultiheadAttention(model_dimension, num_heads=4, batch_first=True)
cross_output, cross_weights = cross_attention(
    label_queries, acoustic_tokens, acoustic_tokens,
    key_padding_mask=~frame_mask,
)

assert self_output.shape == acoustic_tokens.shape
assert cross_output.shape == (acoustic_frames.shape[0], 8, model_dimension)
assert cross_weights.shape == (acoustic_frames.shape[0], 8, acoustic_frames.shape[1])
assert torch.allclose(cross_weights.masked_select(~frame_mask[:, None, :]), torch.zeros_like(
    cross_weights.masked_select(~frame_mask[:, None, :])
))
print({"self": tuple(self_output.shape), "cross": tuple(cross_output.shape)})
```

</details>

The label queries are teacher-forced ground-truth embeddings and are used only to expose the cross-attention interface. A generative decoder must shift targets so a position never receives the label it is supposed to predict.

### **Causal, Padding, and Structural Masks** {#causal-padding-structural-masks}

Masks encode which information paths are legal.

- A **padding mask** prevents real queries from reading artificial padded keys. It is usually `[B,T_k]` and broadcasts across heads and query positions.
- A **causal mask** prevents position $t$ from reading keys at positions $j>t$. It is lower triangular and preserves autoregressive factorization.
- A **structural mask** permits only local windows, blocks, graph edges, modalities, or application-defined relationships.

For additive mask $M_{ij}$,

$$
M_{ij}=\begin{cases}
0,&\text{interaction allowed},\\
-\infty,&\text{interaction forbidden}.
\end{cases}
$$

Masking must occur **before** softmax. Multiplying probabilities by zero afterward leaves the remaining row sum below one unless it is renormalized, and masked positions may already have influenced numerical maxima. If an entire query row is masked, softmax of all $-\infty$ is undefined; such queries should be removed, given a valid sentinel path, or zeroed explicitly.

Training targets must also be shifted correctly. A causal mask cannot prevent leakage if token $y_t$ itself is placed in the input position used to predict $y_t$. Decoder input should contain a start token followed by $y_{<t}$.

<details>
<summary><strong>PyTorch: verify padding and causal information boundaries</strong></summary>

```python
padding_weights = manual_mha.last_weights
masked_padding_weights = padding_weights.masked_select(~frame_mask[:, None, None, :])
assert torch.all(masked_padding_weights == 0)

start = torch.full((word_labels.shape[0], 1), 2, dtype=torch.long)
decoder_inputs = torch.cat([start, word_labels[:, :-1]], dim=1)
decoder_tokens = label_embedding(decoder_inputs)
causal_mask = torch.triu(torch.ones(8, 8, dtype=torch.bool), diagonal=1)
causal_attention = nn.MultiheadAttention(model_dimension, 4, batch_first=True)
causal_output, causal_weights = causal_attention(
    decoder_tokens, decoder_tokens, decoder_tokens,
    attn_mask=causal_mask,
)

future_entries = causal_weights[:, causal_mask]
assert causal_output.shape == decoder_tokens.shape
assert torch.all(future_entries == 0)
assert torch.equal(decoder_inputs[:, 1:], word_labels[:, :-1])
print({"padding weights checked": masked_padding_weights.numel(),
       "future weights checked": future_entries.numel()})
```

</details>

A reliable test changes forbidden inputs while holding allowed inputs fixed and verifies that earlier outputs do not change. Inspecting a mask tensor alone cannot catch target shifting, broadcasting, or cache-position errors.

### **Positional Encoding and RoPE** {#positional-encoding-rope}

Without positional information, self-attention treats tokens as a set. Absolute sinusoidal encoding adds deterministic vectors

$$
PE_{p,2i}=\sin\left(p/10000^{2i/D}\right),
\qquad
PE_{p,2i+1}=\cos\left(p/10000^{2i/D}\right)
$$

to token embeddings. Different frequencies let linear combinations express relative offsets, and no learned table is required. Learned absolute embeddings can adapt to the training range but need an extrapolation policy beyond it.

Relative-position methods alter attention logits according to distance. ALiBi adds a head-specific linear distance bias. **Rotary Position Embedding (RoPE)** rotates each pair of query/key features by a position-dependent angle. For one two-dimensional pair,

$$
R(m\theta)^\top R(n\theta)=R((n-m)\theta),
$$

so the dot product between a query at $m$ and key at $n$ depends on relative offset $n-m$ while each vector still receives an absolute rotation. Different feature pairs use different frequencies.

![RoPE rotates query and key feature pairs so their dot product exposes relative position.](assets/dl09-rope.svg){fig-align="center" width="76%" fig-alt="Rotary position embedding diagram showing position-dependent query and key rotations and a relative-angle dot product."}

*Image source: local educational diagram based on [Su et al., RoFormer: Enhanced Transformer with Rotary Position Embedding](https://arxiv.org/abs/2104.09864).*

RoPE does not make context extrapolation automatically reliable. The frequency spectrum, maximum training length, scaling method, precision, and task all matter. Position interpolation and frequency rescaling modify behavior and must be evaluated on lengths beyond training rather than assumed to work.

<details>
<summary><strong>PyTorch: apply sinusoidal encoding and verify RoPE relative shifts</strong></summary>

```python
def sinusoidal_encoding(length, dimension):
    positions = torch.arange(length, dtype=torch.float32).unsqueeze(1)
    frequencies = torch.exp(
        torch.arange(0, dimension, 2, dtype=torch.float32) * (-math.log(10000.0) / dimension)
    )
    encoding = torch.zeros(length, dimension)
    encoding[:, 0::2] = torch.sin(positions * frequencies)
    encoding[:, 1::2] = torch.cos(positions * frequencies)
    return encoding


def apply_rope(x, positions):
    dimension = x.shape[-1]
    frequencies = 1.0 / (10000.0 ** (torch.arange(0, dimension, 2) / dimension))
    angles = positions.float().unsqueeze(-1) * frequencies
    even, odd = x[..., 0::2], x[..., 1::2]
    rotated_even = even * angles.cos() - odd * angles.sin()
    rotated_odd = even * angles.sin() + odd * angles.cos()
    return torch.stack((rotated_even, rotated_odd), dim=-1).flatten(-2)


ordered_tokens = acoustic_tokens[0, :frame_lengths[0]]
absolute_tokens = ordered_tokens + sinusoidal_encoding(len(ordered_tokens), model_dimension)
assert not torch.allclose(absolute_tokens[0], absolute_tokens[1])

q = ordered_tokens[3:4]
k = ordered_tokens[11:12]
score_a = (apply_rope(q, torch.tensor([3])) * apply_rope(k, torch.tensor([11]))).sum()
score_b = (apply_rope(q, torch.tensor([8])) * apply_rope(k, torch.tensor([16]))).sum()
assert torch.allclose(score_a, score_b, atol=1e-5)
print({"sequence length": len(ordered_tokens), "relative offset": 8,
       "shift-invariant RoPE score": score_a.item()})
```

</details>

The equality holds because both positions shift by five while their relative offset remains eight. In multi-head attention, RoPE is applied to query and key projections, not usually to values.

### **Transformer Encoder and Decoder Blocks** {#transformer-encoder-decoder-blocks}

The original Transformer is an encoder-decoder architecture. Each encoder block applies self-attention and a position-wise feed-forward network (FFN). Each decoder block adds masked self-attention, cross-attention to encoder memory, and an FFN. Residual connections and normalization surround these sublayers.

For encoder input $X\in\mathbb{R}^{B\times T_s\times D}$ and decoder state $Y\in\mathbb{R}^{B\times T_t\times D}$:

- encoder self-attention uses $Q=K=V=X$ and returns length $T_s$;
- decoder causal self-attention uses $Q=K=V=Y$ and returns length $T_t$;
- cross-attention uses $Q=Y$, $K=V=X_{enc}$ and returns length $T_t$.

![The Transformer combines stacked encoder self-attention blocks with a causally masked decoder and encoder-decoder attention.](assets/dl09-transformer.svg){fig-align="center" width="72%" fig-alt="Transformer encoder-decoder architecture with self-attention, cross-attention, feed-forward networks, residual paths, and positional encoding."}

*Image source: [Dive into Deep Learning, The Transformer Architecture](https://d2l.ai/chapter_attention-mechanisms-and-transformers/transformer.html), CC BY-SA 4.0.*

The decoder receives shifted targets during training. At output step $u$, the model sees start token and labels $y_{<u}$, then predicts $y_u$. Teacher forcing allows all target positions to train in parallel under a causal mask, even though inference generates them one at a time.

<details>
<summary><strong>PyTorch: build an acoustic encoder and label decoder</strong></summary>

```python
class TinySpeechTransformer(nn.Module):
    def __init__(self, input_size=129, dimension=32, heads=4):
        super().__init__()
        self.dimension = dimension
        self.input_projection = nn.Linear(input_size, dimension)
        self.label_embedding = nn.Embedding(3, dimension)
        encoder_layer = nn.TransformerEncoderLayer(
            dimension, heads, dim_feedforward=64, dropout=0.0,
            batch_first=True, norm_first=True,
        )
        decoder_layer = nn.TransformerDecoderLayer(
            dimension, heads, dim_feedforward=64, dropout=0.0,
            batch_first=True, norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=2)
        self.decoder = nn.TransformerDecoder(decoder_layer, num_layers=2)
        self.output = nn.Linear(dimension, 2)

    def forward(self, source, source_mask, target_input):
        source_tokens = self.input_projection(source)
        source_tokens = source_tokens + sinusoidal_encoding(source.shape[1], self.dimension)
        memory = self.encoder(source_tokens, src_key_padding_mask=~source_mask)
        target_tokens = self.label_embedding(target_input)
        target_tokens = target_tokens + sinusoidal_encoding(target_input.shape[1], self.dimension)
        causal = torch.triu(torch.ones(target_input.shape[1], target_input.shape[1], dtype=torch.bool), 1)
        decoded = self.decoder(
            target_tokens, memory, tgt_mask=causal,
            memory_key_padding_mask=~source_mask,
        )
        return self.output(decoded), memory


seed_everything(911)
speech_transformer = TinySpeechTransformer()
teacher_input = torch.cat([
    torch.full((word_labels.shape[0], 1), 2, dtype=torch.long),
    word_labels[:, :-1],
], dim=1)
label_logits, encoder_memory = speech_transformer(acoustic_frames, frame_mask, teacher_input)
sequence_loss = F.cross_entropy(label_logits.flatten(0, 1), word_labels.flatten())
sequence_loss.backward()

assert label_logits.shape == (acoustic_frames.shape[0], 8, 2)
assert encoder_memory.shape == acoustic_tokens.shape
assert all(parameter.grad is not None for parameter in speech_transformer.parameters())
print({"memory": tuple(encoder_memory.shape), "label logits": tuple(label_logits.shape),
       "teacher-forced loss": sequence_loss.item()})
```

</details>

This example verifies the complete information path but does not train a speech recognizer. A real experiment would optimize across the training split and evaluate sequence error on unseen speakers, not infer quality from one backward pass.

### **Residual Paths, Normalization, and Feed-Forward Networks** {#residual-normalization-feed-forward}

Attention mixes information **across positions**. The FFN independently transforms channels at every position using shared weights:

$$
\operatorname{FFN}(x)=W_2\,\phi(W_1x+b_1)+b_2.
$$

The inner dimension is commonly several times $D$, making the FFN a major share of parameters and FLOPs. Gated variants such as GLU, GEGLU, and SwiGLU multiply one branch by a learned gate; they change the channel computation without changing sequence mixing.

Residual connections preserve an identity route and require every sublayer to return dimension $D$. Layer normalization stabilizes feature scale per token. Two arrangements are common:

$$
\text{post-norm: }x_{l+1}=\operatorname{LN}(x_l+F(x_l)),
$$

$$
\text{pre-norm: }x_{l+1}=x_l+F(\operatorname{LN}(x_l)).
$$

Post-norm matches the original Transformer but can be difficult to optimize at depth because every identity path passes through normalization. Pre-norm usually improves gradient flow and is common in deep models, although it changes activation growth and final-normalization requirements. Residual scaling, careful initialization, and normalization variants remain active design choices.

<details>
<summary><strong>PyTorch: compare pre-norm and post-norm gradient paths on audio tokens</strong></summary>

```python
class TransformerSubBlock(nn.Module):
    def __init__(self, dimension=32, heads=4, pre_norm=True):
        super().__init__()
        self.pre_norm = pre_norm
        self.norm1 = nn.LayerNorm(dimension)
        self.norm2 = nn.LayerNorm(dimension)
        self.attention = nn.MultiheadAttention(dimension, heads, batch_first=True)
        self.ffn = nn.Sequential(nn.Linear(dimension, 64), nn.GELU(), nn.Linear(64, dimension))

    def forward(self, x, valid):
        if self.pre_norm:
            normalized = self.norm1(x)
            attended = self.attention(normalized, normalized, normalized,
                                      key_padding_mask=~valid, need_weights=False)[0]
            x = x + attended
            return x + self.ffn(self.norm2(x))
        attended = self.attention(x, x, x, key_padding_mask=~valid, need_weights=False)[0]
        x = self.norm1(x + attended)
        return self.norm2(x + self.ffn(x))


gradient_records = {}
for name, pre_norm in (("pre", True), ("post", False)):
    seed_everything(912)
    block_input = acoustic_tokens[:2].detach().clone().requires_grad_(True)
    block = TransformerSubBlock(pre_norm=pre_norm)
    output = block(block_input, frame_mask[:2])
    loss = output[frame_mask[:2]].pow(2).mean()
    loss.backward()
    gradient_records[name] = block_input.grad.norm().item()
    assert torch.isfinite(block_input.grad).all()

print({"input gradient norms": gradient_records,
       "FFN parameters": sum(p.numel() for p in TransformerSubBlock().ffn.parameters())})
```

</details>

One randomly initialized block cannot establish a universal gradient ranking. The experiment verifies both computational paths and provides the logging pattern required for a depth sweep. Claims about trainability need many layers, controlled initialization, and repeated runs.

### **Encoder-Only, Decoder-Only, and Encoder-Decoder Models** {#encoder-decoder-model-families}

Transformer families differ primarily in their information boundary and training objective.

**Encoder-only** models use bidirectional self-attention over observed inputs. They suit classification, retrieval embeddings, and token-level prediction where the entire input is available. Masked-token pretraining hides selected inputs but the remaining context is bidirectional.

**Decoder-only** models use causal self-attention and optimize autoregressive likelihood

$$
p(x_{1:T})=\prod_{t=1}^{T}p(x_t\mid x_{<t}).
$$

The same interface supports generation and in-context conditioning, but every predicted token must obey causal masking. Prompt tokens are observed context; generated tokens extend that context.

**Encoder-decoder** models condition an autoregressive target on a separately encoded source:

$$
p(y_{1:U}\mid x_{1:T})=\prod_{u=1}^{U}p(y_u\mid y_{<u},x_{1:T}).
$$

They are natural when input and output have different modalities, lengths, or roles, such as speech-to-text and translation. The encoder can process source bidirectionally while the decoder remains causal.

<details>
<summary><strong>PyTorch: train an encoder-only first-word classifier on YESNO</strong></summary>

```python
class SpeechEncoderClassifier(nn.Module):
    def __init__(self, input_size=129, dimension=32, heads=4):
        super().__init__()
        self.projection = nn.Linear(input_size, dimension)
        layer = nn.TransformerEncoderLayer(
            dimension, heads, dim_feedforward=64, dropout=0.1,
            batch_first=True, norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(layer, num_layers=1)
        self.head = nn.Linear(dimension, 2)

    def forward(self, x, valid):
        tokens = self.projection(x) + sinusoidal_encoding(x.shape[1], 32)
        encoded = self.encoder(tokens, src_key_padding_mask=~valid)
        pooled = (encoded * valid.unsqueeze(-1)).sum(1) / valid.sum(1, keepdim=True)
        return self.head(pooled)


@torch.no_grad()
def classifier_accuracy(model, indices):
    model.eval()
    correct = total = 0
    for xb, _, valid, labels in yesno_loader(indices):
        prediction = model(xb, valid).argmax(1)
        correct += int((prediction == labels[:, 0]).sum())
        total += len(labels)
    return correct / total


seed_everything(913)
encoder_classifier = SpeechEncoderClassifier()
optimizer = torch.optim.AdamW(encoder_classifier.parameters(), lr=2e-3, weight_decay=1e-3)
for epoch in range(8):
    encoder_classifier.train()
    for xb, _, valid, labels in yesno_loader(train_idx, shuffle=True, seed=913):
        optimizer.zero_grad(set_to_none=True)
        loss = F.cross_entropy(encoder_classifier(xb, valid), labels[:, 0])
        loss.backward()
        nn.utils.clip_grad_norm_(encoder_classifier.parameters(), 1.0)
        optimizer.step()

validation_accuracy = classifier_accuracy(encoder_classifier, val_idx)
assert 0.0 <= validation_accuracy <= 1.0
print({"encoder-only validation accuracy": validation_accuracy,
       "parameters": sum(p.numel() for p in encoder_classifier.parameters())})
```

</details>

The tiny validation set makes the score highly discrete. Its purpose is to prove that the same normalized audio, mask, and split support end-to-end Transformer training. Family selection should follow the legal context and output structure, not model popularity.

### **KV Cache and Autoregressive Inference** {#kv-cache-autoregressive-inference}

During teacher-forced training, a causal decoder processes all target positions in parallel. During autoregressive inference, it produces one token, appends it, and repeats. A naive implementation recomputes key and value projections for the entire prefix at every step.

A **KV cache** stores the key and value tensors produced by every decoder layer. At step $t+1$, only the new token is projected into $q_{t+1},k_{t+1},v_{t+1}$; the query attends to cached $K_{1:t+1},V_{1:t+1}$. The cache removes repeated projection and repeated processing of old tokens, but it does not remove the new query's dot products against all retained keys.

![A KV cache stores past key and value projections while each new token appends one pair and issues one query.](assets/dl09-kv-cache.svg){fig-align="center" width="78%" fig-alt="Autoregressive decoding diagram with previous key-value projections stored in a persistent cache and one new query attending to them."}

*Image source: local educational diagram derived from the incremental causal-attention computation.*

For $L$ layers, batch size $B$, cached length $T$, key/value head count $H_{kv}$, head dimension $d_h$, and element size $s$ bytes, cache memory is approximately

$$
M_{KV}=2LBTH_{kv}d_hs.
$$

The factor two stores keys and values. Multi-query attention shares one K/V head across query heads; grouped-query attention uses an intermediate number, reducing cache memory and bandwidth. Beam search multiplies or shares cache state carefully, while continuous batching must track sequence-specific positions and eviction.

<details>
<summary><strong>PyTorch: prove cached decoding matches full causal attention</strong></summary>

```python
seed_everything(914)
label_sequence = word_labels[0]
label_tokens = label_embedding(label_sequence.unsqueeze(0))[0]
q_layer = nn.Linear(model_dimension, model_dimension, bias=False)
k_layer = nn.Linear(model_dimension, model_dimension, bias=False)
v_layer = nn.Linear(model_dimension, model_dimension, bias=False)
q_all, k_all, v_all = q_layer(label_tokens), k_layer(label_tokens), v_layer(label_tokens)

full_scores = q_all @ k_all.T / math.sqrt(model_dimension)
full_scores = full_scores.masked_fill(torch.triu(torch.ones(8, 8, dtype=torch.bool), 1), float("-inf"))
full_output = full_scores.softmax(-1) @ v_all

cached_keys, cached_values, cached_outputs = [], [], []
for step in range(len(label_tokens)):
    cached_keys.append(k_all[step:step + 1])
    cached_values.append(v_all[step:step + 1])
    key_cache = torch.cat(cached_keys, dim=0)
    value_cache = torch.cat(cached_values, dim=0)
    step_scores = q_all[step:step + 1] @ key_cache.T / math.sqrt(model_dimension)
    cached_outputs.append(step_scores.softmax(-1) @ value_cache)
cached_output = torch.cat(cached_outputs, dim=0)

assert torch.allclose(full_output, cached_output, atol=1e-6)
cache_elements = 2 * len(label_tokens) * model_dimension
print({"tokens": len(label_tokens), "cached K/V elements": cache_elements,
       "maximum output error": (full_output - cached_output).abs().max().item()})
```

</details>

Real caches also store one tensor per layer and must apply the correct positional encoding to the new query/key. Off-by-one position IDs, mismatched causal masks, stale cache entries, and beam reordering are common inference bugs.

### **FlashAttention and Efficient Attention Principles** {#flashattention-efficient-attention}

Naive attention writes the full score matrix $S=QK^\top$ and probability matrix $P=\operatorname{softmax}(S)$ to high-bandwidth memory (HBM), then reads them again to compute $PV$. On modern accelerators, these memory transfers can dominate arithmetic.

FlashAttention is an **exact, IO-aware** algorithm. It tiles queries, keys, and values into on-chip SRAM, computes one score block at a time, and maintains online softmax statistics. For a row processed across key blocks, keep running maximum $m$, denominator $\ell$, and unnormalized output numerator $o$. When a new block has maximum $m_b$,

$$
m'=\max(m,m_b),
$$

$$
\ell'=e^{m-m'}\ell+\sum_j e^{s_j-m'},
\qquad
o'=e^{m-m'}o+\sum_j e^{s_j-m'}v_j.
$$

After all blocks, the output is $o/\ell$. Rescaling by the new maximum preserves exact stable softmax even though no complete row is resident at once. The backward pass can recompute selected intermediates rather than storing the full probability matrix.

![FlashAttention tiles Q, K, and V into on-chip memory and updates exact online-softmax statistics without materializing the full score matrix in HBM.](assets/dl09-flashattention-tiling.svg){fig-align="center" width="78%" fig-alt="FlashAttention memory diagram with HBM blocks, an SRAM score tile, online softmax state, and exact output."}

*Image source: local educational diagram based on [Dao et al., FlashAttention: Fast and Memory-Efficient Exact Attention with IO-Awareness](https://arxiv.org/abs/2205.14135).*

FlashAttention changes the kernel schedule, not the mathematical complexity of dense all-pairs interaction. Runtime benefits depend on sequence length, head dimension, dtype, hardware, masks, dropout, and kernel availability. PyTorch's `scaled_dot_product_attention` can dispatch to optimized backends; production code should prefer it over a handwritten kernel and verify the selected backend through profiling.

<details>
<summary><strong>PyTorch: reproduce exact attention with tiled online softmax</strong></summary>

```python
def tiled_online_attention(q, k, v, block_size=16):
    outputs = []
    scale = 1.0 / math.sqrt(q.shape[-1])
    for q_start in range(0, len(q), block_size):
        q_block = q[q_start:q_start + block_size]
        running_max = torch.full((len(q_block), 1), float("-inf"))
        running_sum = torch.zeros(len(q_block), 1)
        running_output = torch.zeros(len(q_block), v.shape[-1])
        for k_start in range(0, len(k), block_size):
            k_block = k[k_start:k_start + block_size]
            v_block = v[k_start:k_start + block_size]
            score_block = q_block @ k_block.T * scale
            new_max = torch.maximum(running_max, score_block.amax(-1, keepdim=True))
            old_scale = torch.exp(running_max - new_max)
            probabilities = torch.exp(score_block - new_max)
            running_output = old_scale * running_output + probabilities @ v_block
            running_sum = old_scale * running_sum + probabilities.sum(-1, keepdim=True)
            running_max = new_max
        outputs.append(running_output / running_sum)
    return torch.cat(outputs, dim=0)


attention_input = acoustic_tokens[0, :min(int(frame_lengths[0]), 48)]
q_flash = query_projection(acoustic_frames[0, :len(attention_input)])
k_flash = key_projection(acoustic_frames[0, :len(attention_input)])
v_flash = value_projection(acoustic_frames[0, :len(attention_input)])
reference = (q_flash @ k_flash.T / math.sqrt(model_dimension)).softmax(-1) @ v_flash
tiled = tiled_online_attention(q_flash, k_flash, v_flash, block_size=16)

assert torch.allclose(reference, tiled, atol=1e-5)
assert 16 * 16 < len(q_flash) * len(k_flash) or len(q_flash) <= 16
print({"sequence length": len(q_flash),
       "full score elements": len(q_flash) * len(k_flash),
       "largest teaching tile": min(16, len(q_flash)) * min(16, len(k_flash)),
       "maximum error": (reference - tiled).abs().max().item()})
```

</details>

The loop demonstrates online-softmax algebra but is slower than ordinary PyTorch because Python launches many small operations. FlashAttention's value comes from a fused hardware implementation and a carefully designed IO schedule.

### **Transformers Beyond Text** {#transformers-beyond-text}

Attention requires tokens, not words. The tokenization step defines the geometry that the Transformer will model.

- **Vision:** images become fixed patches, hierarchical windows, region proposals, or learned visual tokens. Two-dimensional position and multiscale structure matter.
- **Audio:** waveform samples, spectrogram frames, or time-frequency patches become tokens. Long recordings often require subsampling, local front ends, or streaming chunks.
- **Time series:** each time step may contain many variables, or variables themselves may become tokens. Missingness, irregular timestamps, and causal forecasting boundaries require explicit encoding.
- **Multimodal systems:** modality-specific encoders produce tokens that interact through cross-attention, shared self-attention, or a small latent bottleneck.

The same attention equation does not make modalities interchangeable. A text token is discrete and semantically learned; an image patch has two-dimensional neighborhood; an acoustic frame represents overlapping windows; a sensor event may have irregular elapsed time. Position, augmentation, masking objective, and output head must reflect these differences.

For audio, reducing token count is often essential. Grouping $P$ consecutive frames reduces length from $T$ to approximately $T/P$ and full-attention score count by roughly $P^2$, but also removes temporal resolution. A convolutional subsampler can learn this compression; fixed concatenation makes the trade-off explicit.

<details>
<summary><strong>PyTorch: convert YESNO spectrogram frames into acoustic patches</strong></summary>

```python
patch_size = 4
single_audio = acoustic_frames[0, :frame_lengths[0]]
usable_length = (len(single_audio) // patch_size) * patch_size
frame_patches = single_audio[:usable_length].reshape(-1, patch_size * 129)
patch_projection = nn.Linear(patch_size * 129, model_dimension)
audio_patch_tokens = patch_projection(frame_patches)

original_score_elements = usable_length ** 2
patched_score_elements = len(audio_patch_tokens) ** 2
assert audio_patch_tokens.shape == (usable_length // patch_size, model_dimension)
assert original_score_elements // patched_score_elements == patch_size ** 2
print({"original frames": usable_length, "patch tokens": len(audio_patch_tokens),
       "attention score reduction": original_score_elements / patched_score_elements})
```

</details>

This operation cuts score count by $16\times$ for patch size four, but the model can no longer attend to individual frames before patch projection. Efficient tokenization and efficient attention solve different parts of the cost problem and should be evaluated together.

### **Chapter Comparison and Summary** {#chapter-comparison-summary}

Attention is a content-addressed aggregation operator. The Transformer becomes a complete architecture only after combining it with positional structure, masks, channel-wise nonlinear computation, residual optimization paths, and a task-specific information boundary.

| Component or family | Main role | Tensor/cost focus | Characteristic failure |
|---|---|---|---|
| query, key, value | separate retrieval request, address, and content | $Q:[B,T_q,D]$, $K,V:[B,T_k,D]$ | treating attention weights as causal explanations |
| scaled dot product | stable content similarity and pooling | scores `[B,H,T_q,T_k]` | missing $\sqrt{d_k}$ scaling or unstable masking |
| multi-head attention | multiple learned interaction subspaces | reshape `[B,T,D]` to `[B,H,T,d_h]` | silent transpose/view error or redundant heads |
| self-attention | interactions within one sequence | $O(T^2D)$ dense interaction | no order without position information |
| cross-attention | one sequence reads another | $O(T_qT_kD)$ | confusing query and memory lengths/masks |
| padding/causal mask | enforce legal information paths | broadcast before softmax | target leakage or fully masked rows |
| sinusoidal/RoPE | inject order and relative offsets | position-frequency pairs | assuming extrapolation without long-length tests |
| encoder block | bidirectional source representation | self-attention plus FFN | padding contamination |
| decoder block | causal generation, optionally source-conditioned | causal self-attention plus optional cross-attention | unshifted targets or cache-position errors |
| pre-/post-norm | control residual optimization path | per-token feature statistics | extrapolating one-block behavior to deep stacks |
| encoder-only | full-context representation | one observed sequence | invalid for strict causal deployment |
| decoder-only | autoregressive modeling | growing context and cache | quadratic context interaction and error accumulation |
| encoder-decoder | conditional sequence generation | separate source/target lengths | bottleneck in cross-modal alignment or decoding |
| KV cache | reuse past key/value projections | $2LBTH_{kv}d_hs$ bytes | memory growth, stale cache, beam reordering |
| FlashAttention | reduce dense-attention HBM traffic | tiled exact online softmax | confusing IO savings with subquadratic math |

The YESNO experiment provides one continuous path through the chapter:

1. Convert official waveforms into a fixed, training-normalized acoustic sequence and explicit padding mask.
2. Use one real frame as a query over the other frames to expose Q/K/V semantics.
3. Scale dot products, split heads, and verify every tensor shape.
4. Let label queries cross-attend to acoustic memory while preserving source padding boundaries.
5. Add causal and positional structure, then assemble a full teacher-forced encoder-decoder.
6. Train an encoder-only classifier on the same split to connect mechanism to an end-to-end objective.
7. Prove incremental KV-cached decoding matches full causal attention.
8. Prove tiled online softmax matches naive dense attention exactly.
9. Reduce audio sequence length through explicit patching and account for the lost resolution.

The practical selection rule is not “use a Transformer for every sequence.” Use attention when direct content-dependent interaction is worth its memory and compute. Choose encoder-only, decoder-only, or encoder-decoder structure from the legal context and output factorization. Measure tokenization, score matrix size, cache memory, kernel backend, and deployment latency together. Chapter 10 next considers graph neural networks, where permitted interactions are defined by relational structure rather than a dense sequence.